# Ionosphere Detection using Adaline Neural Network

#### Author 1
- **Name:** João Pedro Fernandes de Aquino
- **GitHub:** https://github.com/Joaof14

#### Author 2
- **Name:** Anderson Carlos da Silva Morais
- **GitHub:** https://github.com/AndersonCSM






## Project Steps
1. Data exploration and quality checks;
2. Validate modeling assumptions;
3. Feature selection;
4. Normalization or scaling and Data split;
5. Model fitting (adjustment);
6. Assessment;
7. Coefficient or feature-importance analysis;
8. Residual/error analysis;
9.  Report.

<details open>
  <summary><h2 style="display: inline;">Sumário do Projeto (PT-BR)</h2></summary>

O monitoramento contínuo da ionosfera é crucial para a confiabilidade de sistemas de telecomunicações, navegação GPS e para a previsão de tempestades geomagnéticas. Radares de alta frequência (HF) emitem pulsos e analisam os sinais refletidos para verificar a existência de estruturas eletrônicas coerentes na ionosfera. O Brasil ocupa posição estratégica e desafiadora, situado sobre a Anomalia de Ionização Equatorial, onde a ionosfera apresenta irregularidades intensas e fenômenos como as bolhas de plasma – fatores que afetam diretamente o desempenho de sinais de comunicação e posicionamento. Este projeto utiliza o dataset Ionosphere (UCI Machine Learning Repository), com 351 retornos de radar e 34 atributos contínuos provenientes de autocorrelação de pulsos. O objetivo é construir uma rede Adaline para classificar automaticamente os retornos em "good" (estrutura ionosférica detectada, +1) ou "bad" (ausência de estrutura coerente, −1), contribuindo para sistemas de alerta e diagnóstico da ionosfera.

</details>

<br>

<details>
  <summary><h2 style="display: inline;">Project Summary (EN-US)</h2></summary>

Continuous monitoring of the ionosphere is crucial for the reliability of telecommunications systems, GPS navigation, and the forecasting of geomagnetic storms. High-frequency (HF) radars transmit pulses and analyze the reflected signals to detect coherent electronic structures in the ionosphere. Brazil occupies a strategic and challenging position, situated over the Equatorial Ionization Anomaly, where the ionosphere exhibits intense irregularities and phenomena such as plasma bubbles—factors that directly affect the performance of communication and positioning signals. This project uses the Ionosphere dataset (UCI Machine Learning Repository), with 351 radar returns and 34 continuous attributes derived from pulse autocorrelation. The goal is to build an Adaline network to automatically classify the returns as "good" (ionospheric structure detected, +1) or "bad" (absence of coherent structure, −1), contributing to ionospheric warning and diagnostic systems.

</details>
<br>

## Etapa 1 – Exploração e Verificação da Qualidade dos Dados (PT-BR) / Step 1 – Data Exploration and Quality Checks (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>


**1. Obter e inspecionar o dataset**
- Carregar o dataset diretamente do repositório UCI (`ucimlrepo` ou `ionosphere.data`).
- Criar um DataFrame com as 34 features e a coluna `class`.
- Verificar com `df.info()`: 351 instâncias, 35 colunas, tipos numéricos (`float64`/`int64`) e target categórico (`object`).
- Usar `df.head()` e `df.describe()` para um resumo estatístico.

**2. Recodificar o target**
- Target original: `'g'` (good) e `'b'` (bad).
- Converter para bipolar: `+1` (good) e `−1` (bad).
- Confirmar as quantidades: 225 good e 126 bad.

**3. Verificar a qualidade dos dados**
- **Valores ausentes:** `df.isnull().sum()` – tipicamente nenhum. Se houver, imputar ou remover.
- **Atributos constantes:** Calcular o desvio padrão; remover qualquer preditor com `std ≈ 0`.
- **Duplicatas:** `df.duplicated().sum()` e remoção, se existirem.

**4. Análise univariada**
- Plotar histogramas para cada um dos 34 atributos.
- Plotar boxplots para detectar outliers severos.

**5. Análise bivariada / multivariada**
- Matriz de correlação de Pearson e mapa de calor.
- Correlação preditor–target: gráfico de barras.
- Boxplots por classe (good vs bad) para as features mais correlacionadas.

**6. Documentar as evidências**
- Resumo da qualidade (missings, constantes, outliers).
- Lista de atributos removidos (se houver).
- Matriz de correlação comentada.
- Histogramas e boxplots para o relatório.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>


**1. Obtain and inspect the dataset**
- Load the dataset from the UCI repository (`ucimlrepo` or `ionosphere.data`).
- Create a DataFrame with 34 features and the `class` column.
- Check with `df.info()`: 351 instances, 35 columns, numeric predictors, categorical target.
- Use `df.head()` and `df.describe()` for a statistical summary.

**2. Recode the target**
- Original target: `'g'` (good) and `'b'` (bad).
- Convert to bipolar: `+1` for good, `−1` for bad.
- Confirm counts: 225 good, 126 bad.

**3. Data quality checks**
- **Missing values:** `df.isnull().sum()` – typically none. Handle if any.
- **Constant features:** Remove predictors with `std ≈ 0`.
- **Duplicates:** `df.duplicated().sum()` and removal if needed.

**4. Univariate analysis**
- Histograms for all 34 attributes.
- Boxplots to detect severe outliers.

**5. Bivariate / multivariate analysis**
- Pearson correlation matrix and heatmap.
- Predictor–target correlation bar chart.
- Boxplots by class for the most correlated features.

**6. Document the evidence**
- Quality summary.
- List of removed attributes (if any).
- Annotated correlation matrix.
- Histograms and boxplots saved for the report.
</details>

<br>

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


# Configurações de visualização
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [22]:
# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    auc,
    classification_report,
)

### 1.1 Inspeção Inicial (Initial inspection)

In [23]:
ionosphere = fetch_ucirepo(id=52)

##Obter as features e os targets
X = ionosphere.data.features
y_df = ionosphere.data.targets

In [24]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 34 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Attribute1   351 non-null    int64  
 1   Attribute2   351 non-null    int64  
 2   Attribute3   351 non-null    float64
 3   Attribute4   351 non-null    float64
 4   Attribute5   351 non-null    float64
 5   Attribute6   351 non-null    float64
 6   Attribute7   351 non-null    float64
 7   Attribute8   351 non-null    float64
 8   Attribute9   351 non-null    float64
 9   Attribute10  351 non-null    float64
 10  Attribute11  351 non-null    float64
 11  Attribute12  351 non-null    float64
 12  Attribute13  351 non-null    float64
 13  Attribute14  351 non-null    float64
 14  Attribute15  351 non-null    float64
 15  Attribute16  351 non-null    float64
 16  Attribute17  351 non-null    float64
 17  Attribute18  351 non-null    float64
 18  Attribute19  351 non-null    float64
 19  Attribut

In [25]:
# Metadados relevantes 
print('Missing values:', ionosphere.metadata['has_missing_values'])
print('Feature types:', ionosphere.metadata['feature_types'])

Missing values: no
Feature types: ['Integer', 'Real']


In [26]:
# Descrução das Features
X.describe()

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute25,Attribute26,Attribute27,Attribute28,Attribute29,Attribute30,Attribute31,Attribute32,Attribute33,Attribute34
count,351.000000,351.0,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,...,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000,351.000000
mean,0.891738,0.0,0.641342,0.044372,0.601068,0.115889,0.550095,0.119360,0.511848,0.181345,...,0.396135,-0.071187,0.541641,-0.069538,0.378445,-0.027907,0.352514,-0.003794,0.349364,0.014480
std,0.311155,0.0,0.497708,0.441435,0.519862,0.460810,0.492654,0.520750,0.507066,0.483851,...,0.578451,0.508495,0.516205,0.550025,0.575886,0.507974,0.571483,0.513574,0.522663,0.468337
min,0.000000,0.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,1.000000,0.0,0.472135,-0.064735,0.412660,-0.024795,0.211310,-0.054840,0.087110,-0.048075,...,0.000000,-0.332390,0.286435,-0.443165,0.000000,-0.236885,0.000000,-0.242595,0.000000,-0.165350
50%,1.000000,0.0,0.871110,0.016310,0.809200,0.022800,0.728730,0.014710,0.684210,0.018290,...,0.553890,-0.015050,0.708240,-0.017690,0.496640,0.000000,0.442770,0.000000,0.409560,0.000000
75%,1.000000,0.0,1.000000,0.194185,1.000000,0.334655,0.969240,0.445675,0.953240,0.534195,...,0.905240,0.156765,0.999945,0.153535,0.883465,0.154075,0.857620,0.200120,0.813765,0.171660
max,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [20]:
### Verificar Frequência de cada classe
y_df['Class'].value_counts()

Class
g    225
b    126
Name: count, dtype: int64

### 1.2 Recodificar o Target

In [40]:
y = y_df['Class'].map({'g': 1, 'b': -1}).values
y

array([ 1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1,
       -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,
        1, -1,  1, -1,  1

### 1.3 Verificar a Qualidade dos dados

In [41]:
assert X.isnull().sum().sum() == 0, "Existem missings – tratar antes de prosseguir"

### 1.4

## Etapa 2 – Validação das Premissas do Modelo (PT-BR) / Step 2 – Validate Modeling Assumptions (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Verificar se a relação preditor‑target é aproximadamente linear (condição para a Adaline).
- Analisar a correlação entre preditores; alta correlação não inviabiliza, mas pode exigir regularização.
- Confirmar que o target está codificado corretamente como bipolar (+1/–1).
- Avaliar balanceamento das classes (225 vs 126) e considerar seu impacto no viés do modelo.
- Checar se os dados normalizados atendem à suposição de distribuição aproximadamente simétrica (desejável para convergência rápida da regra Delta).
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Check that the predictor–target relationship is roughly linear (Adaline requirement).
- Analyze predictor–predictor correlation; high correlation does not break the model but may suggest redundancy.
- Confirm target is properly encoded as bipolar (+1/–1).
- Assess class balance (225 vs 126) and its influence on bias.
- Verify that normalized data show approximately symmetric distributions (desirable for fast Delta rule convergence).
</details>

<br>

## Etapa 3 – Seleção de Atributos (PT-BR) / Step 3 – Feature Selection (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Iniciar com todas as 34 features; a remoção só é necessária se houver atributos constantes ou quase constantes identificados na Etapa 1.
- Se desejado, ranquear features pela correlação absoluta com o target.
- Considerar que a Adaline com muitos atributos correlacionados ainda converge, mas pesos podem ser menos interpretáveis.
- Manter um registo das features usadas para a etapa de análise de coeficientes.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Start with all 34 features; removal only if constant/near‑constant attributes were found in Step 1.
- If desired, rank features by absolute correlation with the target.
- Note that Adaline with many correlated features will still converge, though weight interpretation may suffer.
- Keep a record of the final feature set for later coefficient analysis.
</details>

<br>

## Etapa 4 – Normalização/Padronização (PT-BR) / Step 4 – Normalization/Scaling (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Aplicar padronização Z‑score (`StandardScaler`) após a divisão treino‑teste.
- Ajustar o scaler somente no conjunto de treino; transformar treino e teste com os mesmos parâmetros.
- Justificativa algébrica: a regra Delta atualiza pesos com `w = w + η (y - ŷ) x`. Features em escalas diferentes tornam a superfície de erro alongada, exigindo η muito pequeno e causando convergência lenta. A padronização equaliza as variâncias, permitindo um η estável.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Apply Z‑score normalization (`StandardScaler`) after the train‑test split.
- Fit the scaler only on the training set; transform both sets with the fitted parameters.
- Algebraic justification: the Delta rule updates weights via `w = w + η (y - ŷ) x`. Different scales stretch the error surface, requiring a very small η and causing slow convergence. Standardization equalizes variances, enabling a stable learning rate.
</details>

<br>


## Etapa 5 – Divisão dos Dados (PT-BR) / Step 5 – Data Split (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Separar 80% para treino e 20% para teste com `train_test_split` estratificado (`stratify=y`).
- Usar semente aleatória fixa (`random_state=42`) para reprodutibilidade.
- Verificar se a proporção das classes se mantém em ambos os subconjuntos.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Split data into 80% training and 20% testing with stratification (`stratify=y`).
- Use a fixed random seed (`random_state=42`) for reproducibility.
- Confirm that the class distribution is preserved in both subsets.
</details>

<br>


## Etapa 6 – Ajuste do Modelo (Adaline) (PT-BR) / Step 6 – Model Fitting (Adaline) (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Implementar a rede Adaline usando `SGDClassifier` com `loss='squared_error'` e `learning_rate='constant'`.
- Testar múltiplas taxas de aprendizagem: 10⁻⁴, 10⁻³, 10⁻², 5×10⁻², 10⁻¹, com `max_iter=200` e registrando a função de perda (MSE) a cada época.
- Plotar a curva de convergência (MSE vs épocas) para cada η.
- Selecionar o η que proporcionar a convergência mais suave e o menor erro final (considerando uma combinação de erro final e estabilidade).
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Implement Adaline using `SGDClassifier` with `loss='squared_error'` and constant learning rate.
- Test multiple learning rates: 1e‑4, 1e‑3, 1e‑2, 5e‑2, 1e‑1, with `max_iter=200` and record the MSE loss at each epoch.
- Plot convergence curves (MSE vs epochs) for each η.
- Choose the η that yields the smoothest convergence and smallest final loss (balancing final error and stability).
</details>

<br>

## Etapa 7 – Avaliação (PT-BR) / Step 7 – Assessment (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Treinar o modelo final com o melhor η e classificar o conjunto de teste.
- Calcular e apresentar a matriz de confusão.
- Calcular acurácia, precisão, recall e F1‑score (considerando `good` como classe positiva).
- Discutir qual tipo de erro (falso positivo ou falso negativo) é mais crítico no contexto do monitoramento ionosférico.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Train the final model with the best η and predict on the test set.
- Compute and display the confusion matrix.
- Compute accuracy, precision, recall, and F1‑score (with `good` as the positive class).
- Discuss which type of error (false positive or false negative) is more critical for ionospheric monitoring.
</details>

<br>

## Etapa 8 – Análise dos Coeficientes (Pesos) (PT-BR) / Step 8 – Coefficient (Weight) Analysis (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Extrair os pesos (`coef_`) e o viés (`intercept_`) do modelo Adaline treinado.
- Criar um gráfico de barras com os pesos de cada feature.
- Interpretar a magnitude e o sinal: pesos positivos “empurram” a decisão para `good`, negativos para `bad`.
- Comparar os sinais com a correlação preditor‑target obtida na Etapa 1 para verificar consistência.
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Extract the weights (`coef_`) and bias (`intercept_`) from the trained Adaline.
- Plot a bar chart of the feature weights.
- Interpret magnitude and sign: positive weights push the prediction toward `good`, negative toward `bad`.
- Compare weight signs with predictor–target correlations from Step 1 to check consistency.
</details>

<br>


## Etapa 9 – Análise de Erros/Residuais (PT-BR) / Step 9 – Error/Residual Analysis (EN-US)

<details>
  <summary><h3 style="display: inline;">Detalhes (PT-BR)</h3></summary>

- Computar os resíduos (diferença entre target e saída linear contínua) no conjunto de teste.
- Plotar histograma dos resíduos e avaliar sua simetria.
- Verificar a dispersão dos resíduos em função do valor previsto para detectar heterocedasticidade.
- Identificar padrões de erro sistemático (ex.: instâncias mal classificadas concentradas em certas regiões).
</details>

<details>
  <summary><h3 style="display: inline;">Details (EN-US)</h3></summary>

- Compute residuals (difference between target and continuous linear output) on the test set.
- Plot a histogram of residuals and assess symmetry.
- Check residual dispersion vs predicted values to detect heteroscedasticity.
- Identify systematic error patterns (e.g., misclassified instances concentrated in specific regions).
</details>

<br>


<!-- Etapa 10 - PT -->
<details>
  <summary><h2 style="display: inline;">Etapa 10 – Relatório (PT-BR)</h2></summary>


</details>
<br>

<!-- Etapa 10 - EN -->
<details>
  <summary><h2 style="display: inline;">Step 10 – Report (EN-US)</h2></summary>



</details>